## Stage 1: PlasticESM, MLM post-training of ESM-C on the PETadex corpus

Lead     : `Purav Gupta`

Start    : `2026-10-16`

Status   : single-seed run complete and verified; dual-seed run in flight

Files    : `src/petabite/model_module/lora.py`, `src/petabite/mlm_finetune.py`, `scripts/`

This is the lab notebook for Stage 1. Stage 2, the activity labels and linear
probing, lives in the `igem-toronto` repo and is deliberately out of scope here.

## What Stage 1 is

Continue pretraining ESM-C 300M on the PETadex corpus with a masked-LM
objective, training LoRA adapters only and keeping the backbone frozen. The
result, PlasticESM, is the backbone Stage 2 puts an activity head on.

The objective is masked amino acid prediction on **sequence alone**. No
metadata of any kind enters the loss: not HMM family, not signal peptide, not
ProtParam, not taxonomy. That is deliberate, and the reasoning is in the
"metadata" section below.

## The repo did not work when this started

Four independent failures, each verified by running `origin/main` as it stood:

| | |
|---|---|
| `AutoTokenizer.from_pretrained` | `ValueError: Tokenizer class EsmcTokenizer does not exist` |
| `inject_fused_lora` | `RuntimeError: found no fused modules to wrap (qkv=0, ffn=0)` |
| `dump_pip_freeze` | `FileNotFoundError: 'uv'` |
| `config/mlm.yaml` | `eval_split: validation`, a split the corpus does not have |

The second is the one that mattered. `fused_lora.py` dispatched on
`_PyTorchLayerNormLinear` and `_PyTorchLayerNormMLP`; `esm` 3.4.1 renamed those
to `EsmcLayerNormLinear` and `EsmcLayerNormMLP`, so the tree walk matched
nothing. Earlier versions injected zero adapters and trained only the 30 PEFT
`out_proj` adapters while reporting a healthy loss curve. main's guard did at
least refuse to continue, but it blamed `transformer-engine`, which is not
installed.

Separately, saving was broken. PEFT and a hand-rolled mechanism both ran, and
`get_peft_model_state_dict` keeps only keys containing `lora_` **and**
`.default.`. The hand-rolled parameters matched the first test and failed the
second, so a checkpoint held 1,843,200 of 14,745,600 parameters at r=32.

`model_module/lora.py` replaces both with one mechanism over all four
projections per block, owning its own save and load, with three guards: raise
if nothing is adapted, prove each wrapper is bit-identical to the module it
replaced at init, and compare the saved file against the live model.

In [ ]:
# Reproduce the parameter arithmetic the brief reported, which is also the
# check that the adapter layout is understood correctly.
from transformers import AutoModelForMaskedLM
from petabite.model_module.lora import LoraConfig, apply_lora

model = AutoModelForMaskedLM.from_pretrained("biohub/ESMC-300M", trust_remote_code=True)
report = apply_lora(model, LoraConfig(r=32, alpha=64))

out_proj_only = sum(p.numel() for n, p in model.named_parameters()
                    if p.requires_grad and "out_proj" in n)

print(report.summary())
print(f"out_proj alone      : {out_proj_only:,}   (what PEFT used to save)")
print(f"fraction that was lost: {1 - out_proj_only / report.trainable_params:.1%}")

## The corpus

The published corpus is the 18,172,960 90%-identity centroids. One epoch over
all of them is about 57 h on an L40S, and most of the extra data is
near-duplicates, so Stage 1 trains on one member per **60%-identity** cluster
instead: 1,814,296 clusters, about 5.5 h per epoch.

Nothing in the parquet marks which 90%-centroid represents its 60% cluster, so
a rule had to be chosen, and the choice is not cosmetic. Picking each cluster's
largest member gives a subset that is **89.7% Logan** with a median length of
306, because Logan dominates the corpus and its ORFs are called stop to stop.
`build_c60_corpus.py` draws one member per cluster uniformly at random instead,
seeded, which is unbiased with respect to origin and length.

**81 clusters are excluded** because they contain a sequence from the PETai
label set. The exclusion uses the full 1,219-sequence label set rather than one
split's val/test, so it stays valid however Stage 2 re-splits later. No
downstream evaluation sequence is seen during pretraining in any form.

In [ ]:
import json
from pathlib import Path

for seed in (0, 1):
    path = Path(f"/project/6122149/purav/petabite-tokenized/c60-seed{seed}/manifest.json")
    if not path.exists():
        continue
    m = json.loads(path.read_text())
    print(f"seed {m['seed']}: {m['sequences']:,} sequences, {m['tokens']:,} tokens, "
          f"mean {m['mean_tokens']}, {m['clusters_excluded']} clusters excluded")

## Throughput

Measured on one L40S, 15 timed steps after 4 warmup, projected over the corpus.
Tokens per second rather than steps per second, because every knob being tuned
changes how many tokens a step contains.

| | step | tokens/s | peak mem | projected epoch |
|---|---|---|---|---|
| baseline: batch 8 x 1024, fp32, no TF32, unfused AdamW, SAE on, padded to longest | 0.783 s | 10,467 | 23.5 GB | 14.83 h |
| tuned: batch 32 x 512, bf16, TF32, fused AdamW, SAE off | 0.562 s | 26,610 | 31.2 GB | 5.83 h |
| plus length bucketing at batch 48 | 0.524 s | 27,497 | 44.1 GB | 5.65 h |

**2.63x**, 14.8 h to 5.65 h. Most of it is bf16 with TF32 and fused AdamW.
Truncating at 512 rather than 1024 is nearly free in coverage, since the corpus
median is 306 residues. Length bucketing is worth only 3% and costs 13 GB, so
the config keeps batch 32: at this point the run is compute-bound, not
padding-bound.

`compute_sae` is **not** a speed lever. ESM-C defaults it on and it runs a
sparse autoencoder head masked-LM training never reads, so it is switched off
on principle, but measurement found no difference beyond noise.

The single biggest remaining win would be `transformer-engine`, which fuses
LayerNorm+Linear and LayerNorm+MLP. At 27.5k tokens/s this is near 9% MFU on a
362 TFLOPS card. It is not taken because it swaps the fused modules for TE's
classes, which `lora.py` does not wrap, so `apply_lora` would raise. That is
the guard working as intended.

## A 24-hour stall that was not a hang

The first production run sat at `0/9423` for 13 minutes with the GPU at 0%, the
main process pegged on one core and all 8 dataloader workers idle.

`datasets` 5.0.0 changed `dataset[column]` to return a lazy `Column` rather
than a list. transformers reads the lengths that way for `group_by_length`, and
`LengthGroupedSampler` then indexes the result once per comparison while
sorting each megabatch. One lazy access costs 3.4 ms against 0.2 us for a list,
and the sampler needs roughly 23.5M of them: about **23 hours** of building the
batch order, before step one, inside a 12 h job.

Nothing errors. `Column` supports `len()` and indexing, so the sampler accepts
it and is merely 16,700x slower, which presents as a run that looks slow to
start rather than one that is broken.

`read_lengths` reads the Arrow column directly, 0.018 s for the same 1.8M
values. Because that only works in storage order, `split_tokenized` draws its
own permutation and applies it to both the rows and the lengths, and
`read_lengths` refuses a dataset carrying an indices mapping rather than
returning a silently misaligned array.

In [ ]:
import time
from petabite.data_module.tokenized import load_tokenized, read_lengths

corpus = load_tokenized("/project/6122149/purav/petabite-tokenized/c60-seed0")
lazy_column = corpus["length"]
print("type of corpus['length']:", type(lazy_column).__name__)

start = time.time()
_ = [lazy_column[i] for i in range(2000)]
lazy_per_access = (time.time() - start) / 2000

start = time.time()
lengths = read_lengths(corpus)
read_seconds = time.time() - start

start = time.time()
_ = [lengths[i] for i in range(2000)]
list_per_access = (time.time() - start) / 2000

print(f"lazy access  : {lazy_per_access * 1e3:.3f} ms each")
print(f"array access : {list_per_access * 1e6:.3f} us each  "
      f"({lazy_per_access / list_per_access:,.0f}x faster)")
print(f"read_lengths : {read_seconds:.3f} s for {len(lengths):,} values")
print(f"23.5M lazy accesses would be {23.5e6 * lazy_per_access / 3600:.1f} h")

## Run 1: single seed, one epoch

Job 6318656, 1,809,215 sequences, 9,423 steps, **5 h 17 m**.
https://wandb.ai/morphius/petabite-esmc/runs/4dwq33oj

| | |
|---|---|
| train loss | 1.8545 -> **1.678** |
| eval loss | 1.7168 -> **1.6836** |
| adapters | 240 tensors, 14,745,600 parameters, 59 MB |
| verification | completeness, trained, reload: all pass |

Most of the gain is in the first 10% of the epoch (1.854 to 1.715), then it
grinds slowly to 1.678. Eval tracks train with no divergence, so nothing is
overfitting, but it does say a second pass over the same data would buy little.

The up-projections moved from 3.3e-4 after 20 steps to 5.2e-3 at the end, so
gradients reached all 120 adapters, which is what `verify_checkpoint.py`
checks: an untrained adapter round-trips perfectly and is still worthless.

## Two runs collided, and what survived

Jobs 6318367 and 6318656 ran the identical configuration for 5 h 17 m each,
on kn089 and kn026, and **both wrote `outputs/plasticesm-c60-r32`**. Their final
eval losses differ only at the 4th decimal (1.68877 against 1.68867), which is
GPU nondeterminism.

The final artefact is clean: it is timestamped inside the surviving run's
finish, after the other had exited, and that run wrote and verified it as one
atomic sequence. An independent re-verification passed afterwards. The
intermediate checkpoints belonged to whichever job reached them last and were
deleted.

Two guards now exist in `train_mlm.sbatch`: the job id goes in `OUTDIR` by
default, and the script refuses to start when another `plasticesm` job is
queued or running.

## Run 2: two independent draws rather than two epochs

Because the corpus builder draws one member per cluster at random, `--seed 1`
picks a **different** member of most clusters. Loading both draws costs the same
wall time as two epochs and shows the model more distinct protein.

How much more is a real question rather than a rhetorical one, and the answer
is less than it first appears: **48.7% of the 60% clusters are singletons**, so
both seeds are forced to pick the same member there.

| | sequences seen | distinct |
|---|---|---|
| 2 epochs over one draw | 3.63M | 1,814,296 |
| seed0 + seed1 | 3.63M | **2,494,836** (1.38x) |

So a second draw adds 680,540 genuinely new sequences, not 1.8M. Still strictly
better than a second epoch, which adds none, but 1.38x rather than 2x.

In [ ]:
import collections
import pyarrow.parquet as pq

parquet = pq.ParquetFile(
    "/project/6122149/purav/petadex-corpus/centroids90/centroids90.parquet"
)
members_per_cluster = collections.Counter()
for group in range(parquet.num_row_groups):
    for cluster in parquet.read_row_group(group, columns=["c60_id"])["c60_id"].to_pylist():
        members_per_cluster[cluster] += 1

histogram = collections.Counter(members_per_cluster.values())
total = len(members_per_cluster)
# A second independent draw lands on a new member with probability 1 - 1/k.
expected_new = sum(n * (1 - 1.0 / k) for k, n in histogram.items())

print(f"60% clusters : {total:,}")
print(f"  singletons : {histogram[1]:,} ({100 * histogram[1] / total:.1f}%)")
print(f"expected new from a 2nd draw: {expected_new:,.0f}")
print(f"  one draw   : {total:,} distinct")
print(f"  two draws  : {total + expected_new:,.0f} distinct ({(total + expected_new) / total:.2f}x)")

## Mistakes worth recording

Three, because each cost real time and each is now guarded against.

**A duplicate 5-hour run.** I relaunched without checking whether a production
run was already queued, so two identical jobs burned about 5.3 GPU-hours each.
`train_mlm.sbatch` now refuses to start alongside another `plasticesm` job.

**A corpus list that silently halved itself.** `sbatch --export` separates
variables with commas, so `--export=ALL,CORPUS="path0,path1"` arrives inside
the job as `CORPUS=path0`. Job 6334352 therefore trained one corpus while
reporting success; it was caught 3 minutes in only because the step count read
9,423 rather than the expected 18,873. The separator is now a colon, the
resolved list and its length are echoed, and a missing path exits before the
GPU is touched.

**A misdiagnosis of the stall.** The first hypothesis was that reading the
length column through a shuffle indices mapping was slow. Timing
`dataset["length"]` showed 0.00 s and appeared to clear it. That measured the
wrong half: the read is lazy, and the cost is in the 23.5M element accesses
afterwards.

## Why no metadata

ESM-C's corpus layers offer HMM family, signal peptide and ProtParam
descriptors at 100% coverage, and taxonomy and structure more sparsely. None of
the first three are used, for a reason worth stating precisely because it does
not generalise.

HMM family, SignalP and ProtParam are **cheap** functions of the sequence. A
300M model can plausibly learn them unaided, so supplying them adds an
inductive bias at best, not information.

Structure is an **expensive** function of the sequence: a 6B ESMFold2 trunk, 32
sampling steps, 3 loops. The 300M model cannot derive it at any amount of
training, so distilling it transfers computation the student cannot perform.
That asymmetry is why structure remains interesting and the rest does not.

Taxonomy was checked and rejected on coverage: **5.5%** on the 60% subset, not
the 54% an early biased sample suggested. The chain organism to BacDive to
growth temperature is therefore dead at the first link as well as the last.
`melting_point` and `thermostability` are null on all 307,155,746 corpus rows,
so there is no temperature target to distil from at all.

## Stage 2 evaluation: does any of this predict function?

Stage 1 produced an adapted backbone. This section is the measurement that says
whether it bought anything downstream, and it needs the whole section rather
than a number, because the first three answers it gave were wrong and finding
out why is most of the value.

The conclusion, stated first: **no feature set beats sequence length on any
target, substrate stratum or label regime, and PlasticESM never beats stock
ESM-C.** The useful result is not that null but the leakage measurement two
cells below, which says why published numbers on these targets look so much
better.

Two rules apply throughout. The unit of independence is the homology component,
never the sequence, because sequences inside a component share ancestry. And a
backbone result means nothing until it clears sequence length and amino acid
composition, because the very first probe found length beating both 960
dimensional backbones. The shared machinery is in
`src/petabite/eval_module/`.

| script | question |
| --- | --- |
| `scripts/embed_labels.py` | cache both backbones' embeddings once on a GPU |
| `scripts/embed_layers.py` | the same at six depths, via forward hooks |
| `scripts/eval_sweep.py` | every target and feature set, grouped 5-fold |
| `scripts/eval_power.py` | what effect size each label set could detect |
| `scripts/eval_leakage.py` | how much a random split would flatter us |
| `scripts/eval_conditions.py` | sequence property or assay protocol |
| `scripts/eval_robust.py` | stress whatever appears to win |
| `scripts/eval_metric.py` | pooled against fold-aware scoring |
| `scripts/eval_determinism.py` | does the score move with the BLAS threads |

Jobs, in order: `6335942` embeddings, `6336760` first sweep, `6340953`
conditions, `6341022` leakage, `6343514` and `6343616` depth, `6343981`
determinism, `6344624` metric, `6344807` corrected sweep, `6345366` corrected
leakage, `6345431` corrected protocol test.

## The measurement fault that invalidated three earlier answers

Worth putting before the results, because every number in the first version of
this notebook was affected and the same trap is easy to fall into again.

Scores were computed by concatenating the out-of-fold predictions into one
vector and correlating that against the target. This is the usual thing to do
and it is invalid under a strong ridge penalty. As the penalty grows, each
fold's predictions shrink toward *that fold's own training mean*, so the
concatenated vector ranks sequences partly by which fold they landed in. The
homology components here differ enough in mean that the effect dominates.

The controls were signalling it from the first run and were misread as a
finding. Sequence length is one column, so on the `p-np-c4` stratum its score
should be near its direct value. Computed within folds with no model involved
it is **+0.4157**. The pooled metric reported it between **+0.04 and -0.39**
depending on the penalty, and the first sweep printed **-0.062** for length on
`pet_release_conc` and **-0.193** on `km`.

So the headline "length beats the backbones" was right by accident, and the
later "the backbones beat length on p-np-c4" was the artefact talking. Three
retracted figures for that stratum: 0.404, then 0.30, then nothing.

The fix is to z-score each fold's predictions before pooling, which keeps the
ordering each model produced and discards only the fold-specific level and
scale. The per-fold mean is reported alongside as a check, and the two agree.
`stats.grouped_out_of_fold` does this by default now.

Three smaller faults were found on the way, each of which had bent a result:

- **Pre-norm against post-norm.** `TransformerStack.forward` returns
  `self.norm(x), x, ...`, so `last_hidden_state` is after a final LayerNorm
  while a hook on the last block sees it before. Column standard deviation
  26.8 against 0.008. The first layer sweep compared the two as if they were
  the same tensor.
- **Scale against penalty.** A ridge penalty is not scale invariant, and the
  cached features span 0.008 to 26.8. One alpha grid across them compares
  penalties rather than representations. Everything is standardised now.
- **A truncated grid and unstable selection.** The old grid stopped at 10,000,
  which is where the standardised optimum sat, and plain argmax over roughly
  17 training components swung the headline between 0.30 and 0.40. The grid
  reaches 1e6 and selection uses the one-standard-error rule.

Leave-one-component-out is dropped entirely. Its folds are often a single
sequence, which has no within-fold ordering, so it cannot be corrected and
every LOCO number from the first pass is unreliable.

## The sweep: nothing beats sequence length

Grouped 5-fold Spearman, fold-centred, standardised features, human-curated
labels, one value per sequence. Job `6344807`.

| target | n | comp | length | composition | stock | PlasticESM | verdict |
| --- | --- | --- | --- | --- | --- | --- | --- |
| `pet_release_conc` | 157 | 18 | +0.055 | -0.056 | -0.387 | -0.331 | controls by 0.386 |
| `kcat` | 227 | 40 | **+0.343** | +0.133 | -0.025 | -0.217 | controls by 0.368 |
| `km` | 204 | 40 | +0.150 | +0.127 | -0.213 | -0.137 | controls by 0.287 |
| `kcat_over_km` | 216 | 42 | **+0.336** | +0.241 | +0.015 | +0.078 | controls by 0.258 |
| `melting_temperature` | 57 | 8 | +0.275 | -0.389 | +0.211 | -0.158 | controls by 0.064 |

With the 4,898 LLM-extracted labels folded in, which adds families as well as
measurements:

| target | n | comp | length | composition | stock | PlasticESM |
| --- | --- | --- | --- | --- | --- | --- |
| `pet_release_conc` | 188 | 26 | +0.060 | +0.091 | -0.112 | -0.165 |
| `relative_activity` | 82 | 22 | +0.175 | +0.147 | +0.081 | +0.097 |
| `kcat` | 263 | 49 | +0.272 | +0.031 | -0.041 | -0.082 |
| `km` | 226 | 46 | +0.003 | +0.138 | -0.013 | +0.066 |
| `kcat_over_km` | 229 | 45 | +0.260 | +0.313 | +0.052 | -0.009 |
| `weight_loss` | 91 | 27 | +0.090 | +0.128 | +0.074 | +0.044 |
| `melting_temperature` | 68 | 13 | +0.218 | -0.541 | -0.228 | -0.269 |

Twelve verdicts, twelve to the controls. The margins are larger than the first
pass reported, because length's real scores are around +0.34 rather than the
-0.19 to +0.28 the pooled metric printed.

The power simulation rules out the obvious excuse. At 80% power
`pet_release_conc` needs a true correlation of 0.5, so its null says nothing;
but `kcat`, `km` and `kcat_over_km` each detect 0.3, which is inside the range
published probes claim. For those three the null is real. The power machinery
correlates synthetic predictions directly against the target rather than
fitting a model, so it was not touched by the metric fault.

## Homology leakage sets the scale, and only for the high-capacity features

The most useful number here. The same targets, features and ridge, scored two
ways: a plain shuffled 5-fold, which is what sequence-to-function papers
commonly report, and the grouped 5-fold used everywhere else. Both sides are
standardised and fold-centred, so the comparison isolates the split. Job
`6345366`.

| target | n | comp | random stock | grouped stock | stock inflation | length inflation |
| --- | --- | --- | --- | --- | --- | --- |
| `pet_release_conc` | 157 | 18 | **0.626** | -0.387 | **+1.014** | -0.103 |
| `kcat` | 227 | 40 | **0.721** | -0.025 | **+0.746** | +0.177 |
| `kcat_over_km` | 216 | 42 | 0.570 | +0.015 | **+0.555** | +0.073 |
| `melting_temperature` | 57 | 8 | **0.741** | +0.211 | **+0.530** | -0.065 |
| `km` | 204 | 40 | 0.274 | -0.213 | **+0.487** | -0.123 |

Between 90% and 92% of the sequences carrying each target have a relative
elsewhere in the set, which bounds how much leakage can help, and the
embeddings nearly saturate that bound.

The split by feature set is the point. The 960-dimensional embeddings inflate
by 0.49 to 1.01; sequence length inflates by -0.12 to +0.18, which is nothing.
A high-capacity representation can memorise which family a sequence belongs to
and a single length column cannot, so the inflation is a property of
representational capacity rather than of the targets.

Read in the direction that matters: under the conventional protocol this
project would be reporting rho 0.72 on `kcat` and 0.74 on melting temperature,
numbers that look like a working model and would have gone on the wiki without
anyone being dishonest. With relatives removed from the training side the same
model on the same data is at zero.

Two consequences. The nulls above are not evidence that this approach
underperforms published work; they are what published work tends to look like
once homology is controlled. And the v2 split repair in the `igem-toronto`
repo, which cost a day and produced zero leakage at 90%, 50% and 30% identity,
is the only reason any of this can be stated.

## Is the target a property of the sequence, or of the assay?

The labels were the remaining suspect once noise and power were eliminated.
The hypothesis was that each target is mostly assay-protocol variance, since a
sequence carries a median of 4 measurements at different pH, temperature and
substrate and they are collapsed to one median. **It was wrong**, in all four
tests. `scripts/eval_conditions.py`, jobs `6340953` and `6344807`.

**1. Measurement noise is not the limit.** Variance decomposition, which
involves no cross-validation and so is untouched by the metric fault:

| target | within-sequence share | reliability of median | correlation ceiling |
| --- | --- | --- | --- |
| `pet_release_conc` | 18.2% | 0.947 | 0.973 |
| `kcat` | 18.5% | 0.815 | 0.903 |
| `kcat_over_km` | 27.5% | 0.725 | 0.852 |
| `km` | 35.0% | 0.650 | 0.806 |
| `relative_activity` | 50.4% | 0.496 | 0.704 |

Models scored at most +0.21 against ceilings of 0.81 to 0.97. Only
`relative_activity` is genuinely noise-limited, and it should be dropped as a
regression target for that reason.

**2. The protocol carries no detectable signal either.** A conditions-only
ridge over substrate, pH, temperature and log incubation time, scored out of
fold by component with fold centring (job `6345431`):

| target | measurements | rho | CI95 |
| --- | --- | --- | --- |
| `pet_release_conc` | 1448 | -0.234 | [-0.365, 0.215] |
| `kcat` | 544 | -0.157 | [-0.338, 0.084] |
| `kcat_over_km` | 481 | -0.117 | [-0.382, 0.238] |
| `km` | 454 | +0.105 | [-0.091, 0.393] |
| `relative_activity` | 1062 | +0.025 | [0.014, 0.840] |

Every interval spans zero, so the honest reading is no detectable protocol
signal. An earlier version of this notebook claimed the condition model was
*anti*-predictive at around -0.25 and built an argument about substrate and
family confounding on top of it. Those point estimates were produced by the
pooled metric and the claim is withdrawn.

**3. Holding the chemistry fixed does not rescue anything.** The pooled `kcat`
column does mix 40 substrates whose means span three orders of magnitude, from
-0.07 on MHET to +3.19 on acetylcholine, and `kcat` on acetylcholine is 50
measurements over 48 sequences in a single homology component. That is a real
defect in the label set. It is not what was stopping the models:

| variant | n | comp | best control | best model | verdict |
| --- | --- | --- | --- | --- | --- |
| `kcat` @ p-np-c4 | 73 | 22 | +0.279 | +0.282 | dead heat |
| `kcat` @ p-np-c2 | 48 | 20 | +0.407 | +0.067 | controls by 0.340 |
| `kcat_over_km` @ p-np-c4 | 73 | 22 | +0.362 | +0.332 | controls by 0.030 |
| `kcat_over_km` @ p-np-c2 | 48 | 20 | +0.202 | +0.153 | controls by 0.049 |
| `km` @ p-np-c4 | 73 | 22 | +0.439 | +0.226 | controls by 0.213 |
| `km` @ p-np-c2 | 48 | 20 | +0.030 | +0.241 | model by 0.211 |

The single model win, `km` on p-np-c2, is contradicted by the same target on
the other substrate, where the controls win by almost exactly as much. That
inconsistency is what noise looks like.

**4. Regressing the protocol out does not help.** Every residualised target is
null or negative for both backbones while the controls stay positive: `kcat`
-0.018 against +0.218, `kcat_over_km` -0.123 against +0.147, `km` -0.108
against +0.046, `pet_release_conc` -0.222 against +0.138. Residualising hurts
the embeddings specifically, which is unexplained and worth a look before
anyone builds on it.

In [ ]:
# The tables above, regenerated from the job outputs rather than retyped.
import json
from pathlib import Path

LOGS = Path("/project/6122149/purav/petabite/logs")
FEATURES = ("length", "composition", "stock", "plasticesm")
CONTROLS = ("length", "composition")
MODELS = ("stock", "plasticesm")

sweep = json.loads((LOGS / "corrected-6344807" / "sweep.json").read_text())
print("corrected sweep, grouped 5-fold, fold-centred")
header = f"{'target':22} {'n':>4} {'comp':>5}"
print(header + "".join(f"{name:>12}" for name in FEATURES) + "   verdict")
for target, result in sweep.items():
    if result.get("skipped"):
        continue
    scores = {name: result["features"][name]["kfold5"]["spearman"]
              for name in FEATURES}
    control = max(scores[name] for name in CONTROLS)
    model = max(scores[name] for name in MODELS)
    verdict = ("controls" if control >= model else "model")
    print(f"{target:22} {result['n_sequences']:>4} {result['n_components']:>5}"
          + "".join(f"{scores[name]:>+12.3f}" for name in FEATURES)
          + f"   {verdict} by {abs(control - model):.3f}")

print()
leakage = json.loads((LOGS / "corrected-6344807" / "leakage.json").read_text())
print("leakage: random split against grouped, and who it flatters")
print(f"{'target':22} {'random':>9} {'grouped':>9} {'inflation':>10}  feature")
for target, result in leakage.items():
    if result.get("skipped"):
        continue
    for name in ("stock", "length"):
        entry = result["features"][name]
        print(f"{target:22} {entry['random_spearman']:>+9.3f} "
              f"{entry['grouped_spearman']:>+9.3f} "
              f"{entry['inflation']:>+10.3f}  {name}")

## What the evaluation means for the project

**Stage 1 has no demonstrated downstream value.** PlasticESM does not beat
stock ESM-C anywhere, and on several targets it is worse. The adaptation
produced a backbone that works and not a backbone that helps. More epochs
would be spending compute on a question the evaluation cannot currently
answer in the affirmative.

**Neither backbone extracts usable activity signal from these labels.** Across
7 targets, 2 label regimes, 6 substrate strata and a residualised variant,
nothing clears sequence length. Three targets have adequate power to detect
0.3, so for them the null is a result rather than an absence.

**The leakage measurement is the publishable finding.** It is a methods result
rather than a model result: embeddings inflate by 0.49 to 1.01 under a random
split while sequence length does not inflate at all. It is also the strongest
justification for the v2 split repair.

**PET monomer release cannot adjudicate anything.** 157 sequences over 18
components, one holding 72% of the data, power threshold 0.5. It is the
project objective and the target least able to settle a question.

**The label set's limit is independent families, not measurements.** More
assays on the sequences already present will not help. Measurements on
unrelated enzymes will.

**Structure distillation was correctly deprioritised.** The bottleneck is not
representation quality, and a better encoder cannot fix a target that mixes 40
chemically unrelated reactions.

**Two label changes are indicated**, both in the `igem-toronto` release rather
than here: define `kcat`, `km` and `kcat_over_km` per substrate rather than
pooled, and drop `relative_activity` as a regression target given its 0.704
ceiling. Neither is expected to produce a positive result on its own; they
remove known defects so that a future negative means something.

## Search item 1: ranking variants inside one family. Negative.

The leakage result says the backbones recognise families. The obvious next
question is whether they can order the variants inside one, which is what a
protein engineering campaign actually needs and which family recognition
cannot help with. Job `6345802`, `scripts/eval_within_family.py`.

Component `00855c73` is the only place the data supports the question. It
holds 546 sequences and carries almost all of the dense coverage: 113 PET
release values spanning -3.30 to +4.52 in log10 micromolar, 76
`kcat_over_km`, 65 `kcat`, 52 `relative_activity`, 51 `km`, 47 melting
temperatures. Every earlier measurement collapsed this family to a single
unit of independence, discarding all of it.

Grouping inside the family is by identity cluster, not by sequence: 29
`cluster_90` groups among the 113 PET release sequences, 8 `cluster_50`
groups. Near-identical variants never straddle a split.

**The controls had to change.** Within a scaffold family a model can look
good by tracking drift from the wild type, and `length` is nearly
uninformative here: its direct within-fold correlation against PET release is
-0.146. So two variant-effect controls were added, identity to the family
medoid and substitution plus gap counts from it, both from BLOSUM62 global
alignment. The release records no wild type, so the medoid stands in for it.
Without these the bar would have been degenerate and almost anything would
have cleared it.

Seven hand checks on the alignment code passed before any score was computed:
self-identity exactly 1.0 with zero substitutions, a constructed point
mutation counted as exactly one substitution and zero gaps, a deletion
counted as a gap rather than a substitution.

### Within the family, holding out whole identity clusters

Fold-centred pooled rho, with the per-fold mean in brackets as the
conservative check.

| target | n | groups | length | composition | identity | mutations | stock | PlasticESM | verdict |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| `pet_release_conc` | 113 | 29 | -0.003 | **+0.578** (0.196) | +0.449 | +0.462 | +0.406 (0.129) | +0.344 (0.056) | controls by 0.172 |
| `pet_release_conc` (cluster_50) | 113 | 8 | -0.015 | **+0.624** (0.450) | +0.172 | +0.393 | +0.519 (0.222) | +0.478 (0.187) | controls by 0.105 |
| `kcat_over_km` | 76 | 19 | +0.120 | +0.240 | +0.080 | +0.015 | -0.126 | -0.152 | controls by 0.366 |
| `kcat` | 65 | 20 | -0.098 | +0.042 | +0.083 | +0.019 | -0.045 | -0.069 | controls by 0.128 |
| `km` | 51 | 19 | +0.151 | +0.095 | -0.174 | +0.158 | -0.177 | -0.209 | controls by 0.335 |
| `relative_activity` | 52 | 14 | +0.278 | -0.212 | -0.073 | +0.284 | -0.034 | -0.104 | controls by 0.318 |
| `melting_temperature` | 47 | 10 | -0.067 | -0.137 | +0.140 | -0.085 | +0.160 | +0.260 (0.326) | model by 0.120 |

### Trained on every other component, predicting this family

| target | train n | train comps | length | composition | stock | PlasticESM |
| --- | --- | --- | --- | --- | --- | --- |
| `pet_release_conc` | 44 | 17 | +0.021 | -0.023 | **-0.434** (p 0.003) | **-0.392** (p 0.001) |
| `kcat_over_km` | 140 | 41 | +0.419 | +0.549 | +0.234 | +0.352 |
| `kcat` | 162 | 39 | +0.279 | +0.398 | +0.072 | -0.098 |
| `km` | 153 | 39 | +0.293 | +0.550 | -0.382 | -0.296 |

### Verdict against the acceptance test

Failed. The single apparent model win, PlasticESM on melting temperature at
+0.260 against a best control of +0.140, misses on three counts: the interval
[-0.098, 0.567] contains zero, the permutation p is 0.054 rather than below
0.01, and the power simulation puts 80% power at rho 0.5 for that target, so
a correlation of 0.26 could not have been detected reliably in the first
place. No replication on a second target either, since every other target
goes the other way.

Three things worth keeping from the negative.

**Composition is the best predictor of PET release inside the family**, at
+0.578 and +0.624 under the two groupings, beating both backbones. Amino acid
composition is a 20-number summary with no order information at all, so
whatever drives PET release in this variant series is visible without knowing
the sequence.

**Cross-family prediction is significantly backwards.** Both backbones
anti-predict PET release at -0.434 and -0.392 with permutation p of 0.003 and
0.001 while the controls sit at zero. This is the third time the embeddings
have gone significantly negative out of family, after the residualised
targets and several pooled-metric runs, and the pattern now survives the
metric fix. It is unexplained and it is the only consistent effect the
backbones produce.

**Pooled and per-fold scores disagree by a lot here**, for example composition
at +0.578 pooled against +0.196 per fold. Both are valid and the per-fold
mean is the conservative one. The verdict is the same either way, which is
why the table reports both rather than choosing.

## Open

- Whether the dual-seed run beats the single-seed one, the direct test of
  whether more distinct sequence helps at this scale. Job `6334470`, in
  flight, checkpoint still to verify.
- Why residualising the protocol hurts the embeddings specifically while
  leaving the controls positive. Unexplained.
- Whether a middle layer of ESM-C carries more than the last. The depth sweep
  was run twice and both passes predate the metric fix, so it needs redoing
  before anything is claimed from it.
- Whether substrate-stratified targets change the picture once the label
  release is rebuilt around them.
- Structure distillation remains unbuilt and is explicitly not the next thing
  to build.